# 02 — Preprocessing

Runs preprocessing and writes:
- `data/processed/dataset_selected.csv`
- `artifacts/<run_id>/preprocessors.joblib` and feature/target column lists

Uses `configs/colab_large.yaml` by default (recommended for ~100k rows).

In [8]:
from google.colab import drive

drive.mount('/content/drive')

%cd /content/drive/MyDrive/retrofit

PROJECT_ROOT = '/content/drive/MyDrive/retrofit'
CONFIG = 'configs/colab_large.yaml'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/retrofit


In [9]:
# Install Colab dependencies (includes optional faiss-cpu and xgboost)
!pip -q install -r requirements-colab.txt

# Install this project as an editable package
!pip -q install -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bridge-retrofit (pyproject.toml) ... done


In [10]:
import subprocess
import sys
from pathlib import Path

# Quick diagnostics (helps pinpoint the real cause of exit status 1)
cfg_path = (Path(PROJECT_ROOT) / CONFIG).resolve()
print('CONFIG:', cfg_path, 'exists=', cfg_path.exists())

if cfg_path.exists():
    import yaml
    import pandas as pd

    raw = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
    raw_rel = raw.get('data', {}).get('raw_path')
    if raw_rel:
        raw_path = (Path(PROJECT_ROOT) / str(raw_rel)).resolve()
        print('RAW:', raw_path, 'exists=', raw_path.exists())
        if raw_path.exists() and raw_path.suffix.lower() == '.csv':
            head = pd.read_csv(raw_path, nrows=5)
            print('CSV columns:', list(head.columns))

cmd = [
    sys.executable, '-m', 'bridge_retrofit.cli',
    '--config', str(cfg_path),
    '--project-root', PROJECT_ROOT,
    'preprocess',
]
print('Running:', ' '.join(cmd))

res = subprocess.run(cmd, text=True, capture_output=True, cwd=str(Path(PROJECT_ROOT)))
if res.stdout:
    print(res.stdout)
if res.stderr:
    print(res.stderr, file=sys.stderr)
res.check_returncode()

CONFIG: /content/drive/MyDrive/retrofit/configs/colab_large.yaml exists= True
RAW: /content/drive/MyDrive/retrofit/data/raw/bridge_data.csv exists= True
CSV columns: ['Bridge_Name', 'Bridge_Type', 'Material', 'Age_Years', 'Span_Length_m', 'Number_of_Spans', 'Foundation_Type', 'Load_Capacity_Tonnes', 'Traffic_Volume_VPD', 'Failure_Type', 'Cause_of_Failure', 'Severity', 'Failure_Location', 'Soil_Type', 'Water_Table_Level', 'Flow_Velocity_mps', 'Scour_Depth_m', 'Exposure_Condition', 'Initial_Problem', 'Suggested_Retrofit']
Running: /usr/bin/python3 -m bridge_retrofit.cli --config /content/drive/MyDrive/retrofit/configs/colab_large.yaml --project-root /content/drive/MyDrive/retrofit preprocess
